In [ ]:
import requests
import pandas as pd
from io import StringIO

# Código del país que quieras inspeccionar
country_code = "AU"

# URL con parámetro para limitar a los últimos 7 datos
url = (
    f"https://data.bis.org/topics/CBPOL/BIS%2CWS_CBPOL%2C1.0/M.{country_code}"
    "?view=observations&file_format=csv&format=long"
    "&include=code%2Clabel&lastNObservations=7"
)

response = requests.get(url)

if response.status_code == 200:
    print(f"✅ CSV descargado exitosamente para {country_code}.\n")

    # Leer CSV desde la respuesta
    df = pd.read_csv(StringIO(response.text), skiprows=6)

    # Mostrar las columnas y las primeras filas
    print("🧩 Columnas detectadas:")
    print(df.columns.tolist())
    print("\n🔍 Primeras filas:")
    print(df.head(10))

else:
    print(f"❌ Error al descargar CSV: {response.status_code}")



KeyError: 'Date'

In [14]:
df

,"BIS,WS_CBPOL,1.0",M.AU,M:Monthly,AU:Australia,Per cent per year,Units,1976-06-30,F:Free,Unnamed: 8,A:Normal value,4.97
0,"BIS,WS_CBPOL,1.0",M.AU,M:Monthly,AU:Australia,Per cent per year,Units,1976-07-31,F:Free,NaN,A:Normal value,8.27
1,"BIS,WS_CBPOL,1.0",M.AU,M:Monthly,AU:Australia,Per cent per year,Units,1976-08-31,F:Free,NaN,A:Normal value,7.72
2,"BIS,WS_CBPOL,1.0",M.AU,M:Monthly,AU:Australia,Per cent per year,Units,1976-09-30,F:Free,NaN,A:Normal value,7.91
3,"BIS,WS_CBPOL,1.0",M.AU,M:Monthly,AU:Australia,Per cent per year,Units,1976-10-31,F:Free,NaN,A:Normal value,7.47
4,"BIS,WS_CBPOL,1.0",M.AU,M:Monthly,AU:Australia,Per cent per year,Units,1976-11-30,F:Free,NaN,A:Normal value,4.59
...,...,...,...,...,...,...,...,...,...,...,...
581,"BIS,WS_CBPOL,1.0",M.AU,M:Monthly,AU:Australia,Per cent per year,Units,2024-12-31,F:Free,NaN,A:Normal value,4.35
582,"BIS,WS_CBPOL,1.0",M.AU,M:Monthly,AU:Australia,Per cent per year,Units,2025-01-31,F:Free,NaN,A:Normal value,4.35
583,"BIS,WS_CBPOL,1.0",M.AU,M:Monthly,AU:Australia,Per cent per year,Units,2025-02-28,F:Free,NaN,A:Normal value,4.10
584,"BIS,WS_CBPOL,1.0",M.AU,M:Monthly,AU:Australia,Per cent per year,Units,2025-03-31,F:Free,NaN,A:Normal value,4.10


In [7]:
from datetime import datetime, timedelta
import pandas as pd
from alphacast import Alphacast

class AlphacastEMBICollector:
    def __init__(self, api_key, dataset_id):
        self.api_key = api_key
        self.dataset_id = dataset_id
        self.client = Alphacast(api_key=self.api_key)

    def get_price(self, days_back=4):
        df = self.client.datasets.dataset(self.dataset_id).download_data(format="pandas")

        if df.empty:
            print("⚠️ No se encontraron datos en el dataset.")
            return pd.DataFrame()

        country_col = next((col for col in df.columns if col.lower() in ['country', 'pais', 'país']), None)
        date_col = next((col for col in df.columns if col.lower() == 'date'), None)
        embi_col = "EMBI Global Diversified Subindices"

        if not (country_col and date_col and embi_col in df.columns):
            raise Exception("❌ No se encontraron las columnas necesarias en el dataset.")

        df[date_col] = pd.to_datetime(df[date_col])
        from_date = datetime.utcnow() - timedelta(days=days_back)
        df_filtered = df[df[date_col] >= from_date]

        df_filtered = df_filtered[[country_col, date_col, embi_col]].rename(columns={
            country_col: "country",
            date_col: "time",
            embi_col: "value"
        })

        return df_filtered

In [11]:
from datetime import datetime
import pandas as pd
import traceback

from db_connection.supabase.Client import SupabaseConnection
from data_collectors.alphacast.alphacast_collector import AlphacastEMBICollector

DAYS_BACK = 7
API_KEY = "ak_omAffDqHgquwbsqWtVu7"
DATASET_ID = 5293

LATAM_COUNTRIES = [
    "Argentina", "Bolivia", "Brazil", "Chile", "Colombia", "Costa Rica", "Cuba", "Ecuador",
    "El Salvador", "Guatemala", "Honduras", "Mexico", "Nicaragua", "Panama", "Paraguay",
    "Peru", "Dominican Republic", "Uruguay", "Venezuela", "Puerto Rico"
]

connection = SupabaseConnection()
connection.sign_in_as_collector()

# ✅ Ya no se pasa `name`, solo los argumentos esperados
embi_collector = AlphacastEMBICollector(api_key=API_KEY, dataset_id=DATASET_ID)

try:
    print(f"\n🔄 Descargando datos de los últimos {DAYS_BACK} días desde Alphacast...")
    data_frame = embi_collector.get_price(days_back=DAYS_BACK)

    if len(data_frame) == 0:
        print("⚠️ No se encontraron datos en el período indicado.")
    else:
        print(f"✅ Datos descargados: {len(data_frame)} registros totales.")
        print(f"📅 Fechas disponibles: {data_frame['time'].min()} a {data_frame['time'].max()}")

    for country in LATAM_COUNTRIES:
        df_country = data_frame[data_frame['country'].str.lower() == country.lower()]
        print(f"\n🌎 Procesando país: {country} ({len(df_country)} registros)")

        if df_country.empty:
            print("ℹ️ No se encontraron datos para este país.")
            continue

        last = connection.get_last_by(
            table_name='embi',
            column_name='time',
            filter_by=('country', country)
        )

        if len(last) > 0:
            last_time = last[0]['time']
            print(f"🕒 Último dato en base de datos para {country}: {last_time}")
            filtering = df_country[df_country['time'] > last_time].copy(deep=True)
        else:
            print(f"🆕 No hay datos previos para {country}, insertando todo.")
            filtering = df_country.copy(deep=True)

        if filtering.empty:
            print(f"ℹ️ No hay datos nuevos para insertar para {country}.")
            continue

        filtering['time'] = filtering['time'].astype(str)

        # ✅ Reemplaza NaNs por None para compatibilidad con JSON
        filtering = filtering.where(pd.notnull(filtering), None)

        print(f"⬆️ Insertando {len(filtering)} nuevos registros en Supabase para {country}...")
        connection.insert_dataframe(frame=filtering, table_name='embi')

except Exception as e:
    print('❌ Error al guardar datos EMBI:')
    traceback.print_exc()

print("\n✅ Proceso completo.")



🔄 Descargando datos de los últimos 7 días desde Alphacast...
✅ Datos descargados: 40 registros totales.
📅 Fechas disponibles: 2025-05-22 00:00:00 a 2025-05-23 00:00:00

🌎 Procesando país: Argentina (2 registros)
🕒 Último dato en base de datos para Argentina: 2025-05-23T00:00:00
ℹ️ No hay datos nuevos para insertar para Argentina.

🌎 Procesando país: Bolivia (2 registros)
🕒 Último dato en base de datos para Bolivia: 2025-05-23T00:00:00
ℹ️ No hay datos nuevos para insertar para Bolivia.

🌎 Procesando país: Brazil (2 registros)
🕒 Último dato en base de datos para Brazil: 2025-05-23T00:00:00
ℹ️ No hay datos nuevos para insertar para Brazil.

🌎 Procesando país: Chile (2 registros)
🕒 Último dato en base de datos para Chile: 2025-05-23T00:00:00
ℹ️ No hay datos nuevos para insertar para Chile.

🌎 Procesando país: Colombia (2 registros)
🕒 Último dato en base de datos para Colombia: 2025-05-23T00:00:00
ℹ️ No hay datos nuevos para insertar para Colombia.

🌎 Procesando país: Costa Rica (2 registr

In [1]:
from data_collectors.bis.bis_collector import BankForInternationalSettlements

# Instancia del collector
bis = BankForInternationalSettlements(name="InterestRateTest")

# País a probar
pais_test = "NL"  # Cambia por "SE", "TH", etc. para otros test

# Ejecutar la función get_price para inspeccionar el formato
bis.get_price(country_code=pais_test, days_back=7)



🔍 Columnas recibidas para NL:
['BIS,WS_CBPOL,1.0', 'M.NL', 'M:Monthly', 'NL:Netherlands', 'Per cent per year', 'Units', '1945-03-31', 'F:Free', 'A:Normal value', '3.5']
BIS,WS_CBPOL,1.0 M.NL M:Monthly NL:Netherlands Per cent per year Units 1945-03-31 F:Free A:Normal value  3.5
BIS,WS_CBPOL,1.0 M.NL M:Monthly NL:Netherlands Per cent per year Units 1945-04-30 F:Free A:Normal value  3.5
BIS,WS_CBPOL,1.0 M.NL M:Monthly NL:Netherlands Per cent per year Units 1945-05-31 F:Free A:Normal value  3.5
BIS,WS_CBPOL,1.0 M.NL M:Monthly NL:Netherlands Per cent per year Units 1945-06-30 F:Free A:Normal value  3.5
BIS,WS_CBPOL,1.0 M.NL M:Monthly NL:Netherlands Per cent per year Units 1945-07-31 F:Free A:Normal value  3.5
BIS,WS_CBPOL,1.0 M.NL M:Monthly NL:Netherlands Per cent per year Units 1945-08-31 F:Free A:Normal value  3.5
🔎 Explorando posibles columnas con país:
✅ Posible columna país: 'NL:Netherlands' - Ejemplo: NL:Netherlands
